# **Setup**

In [5]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [6]:
if IS_COLAB or True:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [7]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [8]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on kaggle — storage at: /kaggle/working
Running on kaggle — storage at: /kaggle/working


/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [9]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [10]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [11]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_cosine")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + '_cosine'

In [12]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "cosine",
        "topK": optuna_trial.suggest_int("topK", 50, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [13]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-19 19:47:23,417] A new study created in RDB with name: ItemKNNCFRecommender_cosine


  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1870.23 column/sec. Elapsed time 3.73 sec
  Fold 1/5 - Score: 0.14988027153647518
Similarity column 6969 (100.0%), 1909.35 column/sec. Elapsed time 3.65 sec
  Fold 2/5 - Score: 0.14850847833627104
Similarity column 6969 (100.0%), 1875.34 column/sec. Elapsed time 3.72 sec
  Fold 3/5 - Score: 0.14942135138457902
Similarity column 6969 (100.0%), 1881.20 column/sec. Elapsed time 3.70 sec
  Fold 4/5 - Score: 0.14877529946887297
Similarity column 6969 (100.0%), 1892.18 column/sec. Elapsed time 3.68 sec
  Fold 5/5 - Score: 0.14985350162071007
[I 2025-11-19 19:49:48,468] Trial 0 finished with value: 0.14928778046938165 and parameters: {'topK': 135, 'shrink': 762, 'normalize': False, 'feature_weighting': 'none'}. Best is trial 0 with value: 0.14928778046938165.
Similarity column 6969 (100.0%), 1879.46 column/sec. Elapsed time 3.71 sec
  Fold 1/5 - Score: 0.14104308163844434
Similarity column 6969 (100.0%), 1838.70 column/sec. Elapsed time 3.79 sec
  Fold 2/5 - S

In [15]:
optuna.visualization.plot_optimization_history(optuna_study)

In [16]:
optuna.visualization.plot_param_importances(optuna_study)

In [17]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [20]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "cosine",
        "topK": optuna_trial.suggest_int("topK", 100, 120),
        "shrink": optuna_trial.suggest_int("shrink", 0, 20),
        "normalize": True,
        "feature_weighting": "BM25",
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 1.1, 1.3)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.09, 1)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [21]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=7
)

[I 2025-11-19 23:09:01,195] Using an existing study with name 'ItemKNNCFRecommender_cosine_refined' instead of creating a new one.


  0%|          | 0/7 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1903.28 column/sec. Elapsed time 3.66 sec
  Fold 1/5 - Score: 0.21031979957269917
Similarity column 6969 (100.0%), 1911.69 column/sec. Elapsed time 3.65 sec
  Fold 2/5 - Score: 0.2106403885075836
Similarity column 6969 (100.0%), 1930.05 column/sec. Elapsed time 3.61 sec
  Fold 3/5 - Score: 0.21259641219408365
Similarity column 6969 (100.0%), 1922.32 column/sec. Elapsed time 3.63 sec
  Fold 4/5 - Score: 0.2108735060570194
[I 2025-11-19 23:10:26,024] Trial 14 finished with value: 0.21110752658284646 and parameters: {'topK': 103, 'shrink': 5, 'BM25_k1': 1.298180702347415, 'BM25_b': 0.41365504469119446}. Best is trial 0 with value: 0.2203879822116445.
Similarity column 6969 (100.0%), 1934.85 column/sec. Elapsed time 3.60 sec
  Fold 1/5 - Score: 0.2128049644530417
Similarity column 6969 (100.0%), 1936.11 column/sec. Elapsed time 3.60 sec
  Fold 2/5 - Score: 0.21337606325845213
Similarity column 6969 (100.0%), 1929.13 column/sec. Elapsed time 3.61 sec
  Fold 

In [22]:
optuna.visualization.plot_optimization_history(optuna_study)

In [23]:
optuna.visualization.plot_param_importances(optuna_study)

In [24]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- **1 trial**:
Best Value: 0.22050691227509706
Best Params: {'topK': 111, 'shrink': 4, 'normalize': True, 'feature_weighting': 'BM25', 'BM25_k1': 1.1870283836071653, 'BM25_b': 0.10718675269785805}
- **2 trial**:
Best Value: 0.22039502329275967
Best Params: {'topK': 112, 'shrink': 7, 'BM25_k1': 1.140341733993252, 'BM25_b': 0.09136432416356022}

Best: 0.22050691227509706 params: {'topK': 111, 'shrink': 4, 'normalize': True, 'feature_weighting': 'BM25', 'BM25_k1': 1.1870283836071653, 'BM25_b': 0.10718675269785805}